In [1]:
import os
import sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu


In [2]:
import os, json
from pathlib import Path
import pandas as pd, numpy as np
import statsmodels.formula.api as smf

# dùng chung với các notebook khác
PROJECT_ROOT = Path(os.getcwd()).parent
CONFIG_PATH = PROJECT_ROOT / "src" / "config.json"
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = json.load(f)

FEATURES = Path(cfg["FEATURES"])
RESULTS  = Path(cfg["RESULTS"])

print("FEATURES:", FEATURES)
print("RESULTS :", RESULTS)

wd = pd.read_parquet(FEATURES / "weekly_category.parquet")

OUT_DIR = RESULTS
OUT_DIR.mkdir(exist_ok=True)



FEATURES: D:\STAT3013.Q12_Group01\features
RESULTS : D:\STAT3013.Q12_Group01\results


In [3]:
wd["ln_q"] = np.log(wd["qty"].replace(0,1))
wd["ln_p"] = np.log(wd["avg_price"].replace(0,1))

ols = smf.ols("ln_q ~ ln_p + discount_rate + weekofyear + C(group_id)", data=wd).fit()
print(ols.summary().tables[1])


                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                         -0.5311      0.227     -2.341      0.019      -0.976      -0.086
C(group_id)[T.CHARITABLE CONT]     0.3141      0.956      0.329      0.743      -1.562       2.190
C(group_id)[T.CHEF SHOPPE]         0.7081      0.294      2.406      0.016       0.131       1.286
C(group_id)[T.CNTRL/STORE SUP]    -0.5668      0.520     -1.090      0.276      -1.587       0.454
C(group_id)[T.COSMETICS]           3.3493      0.258     12.960      0.000       2.842       3.856
C(group_id)[T.COUP/STR & MFG]      1.3852      0.296      4.681      0.000       0.805       1.966
C(group_id)[T.DAIRY DELI]          1.6722      0.470      3.561      0.000       0.751       2.594
C(group_id)[T.DELI]                5.7466      0.249     23.057      0.000       5.258       6.236
C(group_id

In [4]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# Chuẩn bị dữ liệu
X = wd[["avg_price","discount_rate","weekofyear"]].copy()
X["avg_price"] = np.log(X["avg_price"].replace(0,1))
X["weekofyear"] = X["weekofyear"] / 52.0   # FIX scale

y = np.log(wd["qty"].replace(0,1)).values.reshape(-1,1)

X_t = torch.tensor(X.values.astype("float32"))
y_t = torch.tensor(y.astype("float32"))

dl = DataLoader(TensorDataset(X_t, y_t), batch_size=512, shuffle=True)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3,128), nn.ReLU(),
            nn.Linear(128,64), nn.ReLU(),
            nn.Linear(64,1)
        )
    def forward(self,x): return self.net(x)

m = MLP()
opt = torch.optim.Adam(m.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

print("Bắt đầu training Neural Network...")
m.train()
# --- FIX: Tăng số epochs từ 10 lên 50 để mô hình hội tụ tốt hơn ---
for epoch in range(50):
    total_loss = 0
    for xb, yb in dl:
        pred = m(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item()
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/50 - Loss: {total_loss/len(dl):.4f}")

# Tính độ co giãn (Elasticity) = d lnQ / d lnP
X_t.requires_grad_(True)
log_Q_hat = m(X_t)
d_lnQ_d_lnP = torch.autograd.grad(
    log_Q_hat, X_t,
    grad_outputs=torch.ones_like(log_Q_hat)
)[0][:,0] # Lấy cột 0 tương ứng với feature đầu tiên là avg_price (ln_p)

wd["neural_eps"] = d_lnQ_d_lnP.detach().numpy()
print("Đã tính xong Neural Elasticity.")
wd["neural_eps"] = d_lnQ_d_lnP.detach().numpy()


Bắt đầu training Neural Network...


Epoch 10/50 - Loss: 24.2822
Epoch 20/50 - Loss: 13.9915


Epoch 30/50 - Loss: 14.4222
Epoch 40/50 - Loss: 12.4119


Epoch 50/50 - Loss: 11.4722
Đã tính xong Neural Elasticity.


In [5]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

X = wd[["avg_price","discount_rate","weekofyear"]].copy()
X["avg_price"] = np.log(X["avg_price"].replace(0,1))
X["weekofyear"] = X["weekofyear"] / 52.0   #  FIX scale

y = np.log(wd["qty"].replace(0,1)).values.reshape(-1,1)

X_t = torch.tensor(X.values.astype("float32"))
y_t = torch.tensor(y.astype("float32"))

dl = DataLoader(TensorDataset(X_t, y_t), batch_size=512, shuffle=True)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3,128), nn.ReLU(),
            nn.Linear(128,64), nn.ReLU(),
            nn.Linear(64,1)
        )
    def forward(self,x): return self.net(x)

m = MLP()
opt = torch.optim.Adam(m.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

m.train()
for _ in range(10):
    for xb, yb in dl:
        pred = m(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad(); loss.backward(); opt.step()

# elasticity = d lnQ / d lnP
X_t.requires_grad_(True)
log_Q_hat = m(X_t)
d_lnQ_d_lnP = torch.autograd.grad(
    log_Q_hat, X_t,
    grad_outputs=torch.ones_like(log_Q_hat)
)[0][:,0]

wd["neural_eps"] = d_lnQ_d_lnP.detach().numpy()


In [6]:
import os, pandas as pd, numpy as np

OUT_DIR = r"D:\STAT3013.Q12_Group01\results"
os.makedirs(OUT_DIR, exist_ok=True)


#  nb05_ols_summary.txt  --- full OLS summary
with open(os.path.join(OUT_DIR, "nb05_ols_summary.txt"), "w", encoding="utf-8") as f:
    f.write(ols.summary().as_text())

print("Saved:", os.path.join(OUT_DIR, "nb05_ols_summary.txt"))


# nb05_ols_params.csv  --- params table
ols_params = pd.DataFrame({
    "coef": ols.params,
    "std_err": ols.bse,
    "t": ols.tvalues,
    "p_value": ols.pvalues,
})
ci = ols.conf_int()
ols_params["ci_low"] = ci[0]
ols_params["ci_high"] = ci[1]
ols_params.index.name = "term"

ols_params.to_csv(os.path.join(OUT_DIR, "nb05_ols_params.csv"))
print("Saved:", os.path.join(OUT_DIR, "nb05_ols_params.csv"))

# nb05_elasticity_by_group.csv
#     --- group-level elasticity:
#         + OLS per-group ln_p coef
#         + Neural mean/std per group
# ------------------------------------------------
# safe logs for per-group OLS
wd_tmp = wd.copy()
wd_tmp["ln_q"] = np.log(wd_tmp["qty"].replace(0, 1))
wd_tmp["ln_p"] = np.log(wd_tmp["avg_price"].replace(0, 1))

group_rows = []
for g, df_g in wd_tmp.groupby("group_id"):
    if len(df_g) < 8:  # too small -> skip
        continue
    try:
        # per-group OLS (no fixed effects because within one group)
        ols_g = smf.ols("ln_q ~ ln_p + discount_rate + weekofyear", data=df_g).fit()
        eps_ols = ols_g.params.get("ln_p", np.nan)
        p_ols   = ols_g.pvalues.get("ln_p", np.nan)
    except Exception:
        eps_ols, p_ols = np.nan, np.nan

    # neural summary
    eps_nn_mean = df_g["neural_eps"].mean() if "neural_eps" in df_g.columns else np.nan
    eps_nn_std  = df_g["neural_eps"].std(ddof=1) if "neural_eps" in df_g.columns else np.nan
    n_obs = len(df_g)

    group_rows.append({
        "group_id": g,
        "n_obs": n_obs,
        "eps_ols_group": eps_ols,
        "pvalue_ols_group": p_ols,
        "eps_neural_mean": eps_nn_mean,
        "eps_neural_std": eps_nn_std
    })

elasticity_by_group = pd.DataFrame(group_rows).sort_values("eps_ols_group")
elasticity_by_group.to_csv(os.path.join(OUT_DIR, "nb05_elasticity_by_group.csv"), index=False)
print("Saved:", os.path.join(OUT_DIR, "nb05_elasticity_by_group.csv"))

# nb05_neural_elasticity.csv
# --- row-level neural epsilon
cols_keep = [
    c for c in ["week_no","group_id","qty","avg_price","discount_rate","weekofyear","neural_eps"]
    if c in wd.columns
]
neural_elasticity = wd[cols_keep].copy()
neural_elasticity.to_csv(os.path.join(OUT_DIR, "nb05_neural_elasticity.csv"), index=False)
print("Saved:", os.path.join(OUT_DIR, "nb05_neural_elasticity.csv"))

elasticity_by_group.head()


Saved:

 D:\STAT3013.Q12_Group01\results\nb05_ols_summary.txt
Saved: D:\STAT3013.Q12_Group01\results\nb05_ols_params.csv
Saved: D:\STAT3013.Q12_Group01\results\nb05_elasticity_by_group.csv
Saved: D:\STAT3013.Q12_Group01\results\nb05_neural_elasticity.csv


,group_id,n_obs,eps_ols_group,pvalue_ols_group,eps_neural_mean,eps_neural_std
4,DELI,48,-5.008689,0.004249,1.879577,0.058002
9,GROCERY,48,-3.233900,0.323210,1.500212,0.041652
17,PRODUCE,48,-3.103814,0.089452,1.387814,0.060053
19,SALAD BAR,48,-2.546601,0.007467,1.776091,0.066905
6,FLORAL,48,-0.907683,0.065164,2.007217,0.035317
